In [26]:
import warnings; warnings.filterwarnings(action='ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from mpl_finance import candlestick_ohlc
from binance import Client

In [32]:
class PreProcessor():
    def __init__(self, start_date="1 Aug, 2021"):
        self.client = Client()
        self.tickers = self.get_tickers()
        self.sample_dict = {}
        for ticker in self.tickers:
            sample = self.get_sample(ticker, start_date=start_date)
            self.preprocess(sample)
            self.sample_dict[ticker] = sample.copy()
            del sample
    
    def get_tickers(self):
        tickers = []
        for info in self.client.futures_mark_price():
            ticker = info['symbol']
            if ticker[-4:] == "USDT": tickers.append(ticker)
        
        return tickers
        
    def get_sample(self, ticker, start_date):
        try:
            klines = np.array(self.client.futures_historical_klines(ticker, Client.KLINE_INTERVAL_1HOUR, start_date))
            sample = pd.DataFrame(klines.reshape(-1, 12), dtype=float, columns=['datetime', 
                                                                            'open', 
                                                                            'high', 
                                                                            'low', 
                                                                            'close', 
                                                                            'volume', 
                                                                            'close time', 
                                                                            'quote asset volume, number of trades', 
                                                                            'number of trades',
                                                                            'taker buy base asset volume', 
                                                                            'taker buy quote asset volume', 
                                                                            'ignore'])
        except:
            print(ticker)
            raise(NameError)
        sample['datetime'] = pd.to_datetime(sample['datetime'], unit='ms')
        sample.set_index('datetime', inplace=True)
        sample = sample[['open', 'high', 'low', 'close', 'volume']]
        return sample
    
    def preprocess(self, sample):
        sample['trade_amount'] = sample['volume']*(sample['high']+sample['low']+sample['close'])/3.
        sample['ropen_7'] = (sample['open'] - sample['low'].rolling(7).min())/(sample['high'].rolling(7).max() - sample['low'].rolling(7).min())
        sample['rhigh_7'] = (sample['high'] - sample['low'].rolling(7).min())/(sample['high'].rolling(7).max() - sample['low'].rolling(7).min())
        sample['rlow_7'] = (sample['low'] - sample['low'].rolling(7).min())/(sample['high'].rolling(7).max() - sample['low'].rolling(7).min())
        sample['rclose_7'] = (sample['close'] - sample['low'].rolling(7).min())/(sample['high'].rolling(7).max() - sample['low'].rolling(7).min())
        sample['rvolume_7'] = (sample['volume'] - sample['volume'].rolling(7).min())/(sample['volume'].rolling(7).max() - sample['volume'].rolling(7).min())

        sample['mopen_7'] = (sample['open'] - sample.shift(7)['close'])/sample.shift(7)['close']
        sample['mhigh_7'] = (sample['high'] - sample.shift(7)['close'])/sample.shift(7)['close']
        sample['mlow_7'] = (sample['low'] - sample.shift(7)['close'])/sample.shift(7)['close']
        sample['mclose_7'] = (sample['close'] - sample.shift(7)['close'])/sample.shift(7)['close']
        sample['mvolume_7'] = (sample['volume'] - sample.shift(7)['volume'])/sample.shift(7)['volume']

        sample['ropen_14'] = (sample['open'] - sample['low'].rolling(14).min())/(sample['high'].rolling(14).max() - sample['low'].rolling(14).min())
        sample['rhigh_14'] = (sample['high'] - sample['low'].rolling(14).min())/(sample['high'].rolling(14).max() - sample['low'].rolling(14).min())
        sample['rlow_14'] = (sample['low'] - sample['low'].rolling(14).min())/(sample['high'].rolling(14).max() - sample['low'].rolling(14).min())
        sample['rclose_14'] = (sample['close'] - sample['low'].rolling(14).min())/(sample['high'].rolling(14).max() - sample['low'].rolling(14).min())
        sample['rvolume_14'] = (sample['volume'] - sample['volume'].rolling(14).min())/(sample['volume'].rolling(14).max() - sample['volume'].rolling(14).min())

        sample['mopen_14'] = (sample['open'] - sample.shift(14)['close'])/sample.shift(14)['close']
        sample['mhigh_14'] = (sample['high'] - sample.shift(14)['close'])/sample.shift(14)['close']
        sample['mlow_14'] = (sample['low'] - sample.shift(14)['close'])/sample.shift(14)['close']
        sample['mclose_14'] = (sample['close'] - sample.shift(14)['close'])/sample.shift(14)['close']
        sample['mvolume_14'] = (sample['volume'] - sample.shift(14)['volume'])/sample.shift(14)['volume']

        sample['ropen_24'] = (sample['open'] - sample['low'].rolling(24).min())/(sample['high'].rolling(24).max() - sample['low'].rolling(24).min())
        sample['rhigh_24'] = (sample['high'] - sample['low'].rolling(24).min())/(sample['high'].rolling(24).max() - sample['low'].rolling(24).min())
        sample['rlow_24'] = (sample['low'] - sample['low'].rolling(24).min())/(sample['high'].rolling(24).max() - sample['low'].rolling(24).min())
        sample['rclose_24'] = (sample['close'] - sample['low'].rolling(24).min())/(sample['high'].rolling(24).max() - sample['low'].rolling(24).min())
        sample['rvolume_24'] = (sample['volume'] - sample['volume'].rolling(24).min())/(sample['volume'].rolling(24).max() - sample['volume'].rolling(24).min())

        sample['mopen_24'] = (sample['open'] - sample.shift(24)['close'])/sample.shift(24)['close']
        sample['mhigh_24'] = (sample['high'] - sample.shift(24)['close'])/sample.shift(24)['close']
        sample['mlow_24'] = (sample['low'] - sample.shift(24)['close'])/sample.shift(24)['close']
        sample['mclose_24'] = (sample['close'] - sample.shift(24)['close'])/sample.shift(24)['close']
        sample['mvolume_24'] = (sample['volume'] - sample.shift(24)['volume'])/sample.shift(24)['volume']

        sample['ropen_168'] = (sample['open'] - sample['low'].rolling(168).min())/(sample['high'].rolling(168).max() - sample['low'].rolling(168).min())
        sample['rhigh_168'] = (sample['high'] - sample['low'].rolling(168).min())/(sample['high'].rolling(168).max() - sample['low'].rolling(168).min())
        sample['rlow_168'] = (sample['low'] - sample['low'].rolling(168).min())/(sample['high'].rolling(168).max() - sample['low'].rolling(168).min())
        sample['rclose_168'] = (sample['close'] - sample['low'].rolling(168).min())/(sample['high'].rolling(168).max() - sample['low'].rolling(168).min())
        sample['rvolume_168'] = (sample['volume'] - sample['volume'].rolling(168).min())/(sample['volume'].rolling(168).max() - sample['volume'].rolling(168).min())

        sample['mopen_168'] = (sample['open'] - sample.shift(168)['close'])/sample.shift(168)['close']
        sample['mhigh_168'] = (sample['high'] - sample.shift(168)['close'])/sample.shift(168)['close']
        sample['mlow_168'] = (sample['low'] - sample.shift(168)['close'])/sample.shift(168)['close']
        sample['mclose_168'] = (sample['close'] - sample.shift(168)['close'])/sample.shift(168)['close']
        sample['mvolume_168'] = (sample['volume'] - sample.shift(168)['volume'])/sample.shift(168)['volume']

        sample['ropen_1500'] = (sample['open'] - sample['low'].rolling(1500).min())/(sample['high'].rolling(1500).max() - sample['low'].rolling(1500).min())
        sample['rhigh_1500'] = (sample['high'] - sample['low'].rolling(1500).min())/(sample['high'].rolling(1500).max() - sample['low'].rolling(1500).min())
        sample['rlow_1500'] = (sample['low'] - sample['low'].rolling(1500).min())/(sample['high'].rolling(1500).max() - sample['low'].rolling(1500).min())
        sample['rclose_1500'] = (sample['close'] - sample['low'].rolling(1500).min())/(sample['high'].rolling(1500).max() - sample['low'].rolling(1500).min())
        sample['rvolume_1500'] = (sample['volume'] - sample['volume'].rolling(1500).min())/(sample['volume'].rolling(1500).max() - sample['volume'].rolling(1500).min())

        sample['mopen_1500'] = (sample['open'] - sample.shift(1500)['close'])/sample.shift(1500)['close']
        sample['mhigh_1500'] = (sample['high'] - sample.shift(1500)['close'])/sample.shift(1500)['close']
        sample['mlow_1500'] = (sample['low'] - sample.shift(1500)['close'])/sample.shift(1500)['close']
        sample['mclose_1500'] = (sample['close'] - sample.shift(1500)['close'])/sample.shift(1500)['close']
        sample['mvolume_1500'] = (sample['volume'] - sample.shift(1500)['volume'])/sample.shift(1500)['volume']

        sample['noise'] = 1. - abs(sample['close'] - sample['open'])/(sample['high'] - sample['low'])
        sample['noise7'] = sample['noise'].rolling(7).mean()
        sample['noise14'] = sample['noise'].rolling(14).mean()
        sample['noise24'] = sample['noise'].rolling(24).mean()
        sample['noise168'] = sample['noise'].rolling(168).mean()
        sample['noise1500'] = sample['noise'].rolling(1500).mean()

        sample['reward'] = 1. + sample['close'].pct_change()

        sample.dropna(inplace=True)

In [34]:
processor = PreProcessor()

ROSEUSDT


NameError: 